In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# ============================================================
# 1. Configuration
# ============================================================

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"

TARGET = "loan_paid_back"
ID_COL = "id"
SENSITIVE_COL = "gender"

RANDOM_STATE = 42

# True = only use part of training data to quickly check whether code runs
FAST_MODE = True
SAMPLE_SIZE = 100_000


# ============================================================
# 2. Load data
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())


# ============================================================
# 3. Define X and y
# ============================================================

# Keep gender separately for future fairness analysis
gender = train[SENSITIVE_COL].copy()

# For the main ML model:
# - remove target
# - remove ID
# - temporarily exclude gender
X = train.drop(
    columns=[
        TARGET,
        ID_COL,
        SENSITIVE_COL
    ]
).copy()

y = train[TARGET].astype(int).copy()

X_test = test.drop(
    columns=[
        ID_COL,
        SENSITIVE_COL
    ]
).copy()


# ============================================================
# 4. Optional fast mode
# ============================================================

if FAST_MODE:
    X, _, y, _ = train_test_split(
        X,
        y,
        train_size=min(SAMPLE_SIZE, len(X)),
        stratify=y,
        random_state=RANDOM_STATE
    )

    print("\nFAST MODE ON")
    print("Training sample size:", len(X))


# ============================================================
# 5. Detect numerical / categorical columns
# ============================================================

categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("\nNumerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)


# ============================================================
# 6. Train / validation split
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTrain split:", X_train.shape)
print("Validation split:", X_valid.shape)


# ============================================================
# 7. Preprocessing for RF / LightGBM
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_cols
        ),
        (
            "numerical",
            "passthrough",
            numerical_cols
        )
    ]
)


# ============================================================
# 8. Random Forest
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", rf_model)
    ]
)

print("\n====================================")
print("Training Random Forest...")
print("====================================")

rf_pipeline.fit(
    X_train,
    y_train
)

rf_pred = rf_pipeline.predict_proba(
    X_valid
)[:, 1]

rf_auc = roc_auc_score(
    y_valid,
    rf_pred
)

print(f"Random Forest AUC: {rf_auc:.6f}")


# ============================================================
# 9. LightGBM
# ============================================================

lgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=5.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", lgbm_model)
    ]
)

print("\n====================================")
print("Training LightGBM...")
print("====================================")

lgbm_pipeline.fit(
    X_train,
    y_train
)

lgbm_pred = lgbm_pipeline.predict_proba(
    X_valid
)[:, 1]

lgbm_auc = roc_auc_score(
    y_valid,
    lgbm_pred
)

print(f"LightGBM AUC: {lgbm_auc:.6f}")


# ============================================================
# 10. CatBoost
# ============================================================

# CatBoost can directly handle categorical variables,
# so we do NOT apply OneHotEncoder.

catboost_model = CatBoostClassifier(
    iterations=300,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100
)

print("\n====================================")
print("Training CatBoost...")
print("====================================")

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols
)

catboost_pred = catboost_model.predict_proba(
    X_valid
)[:, 1]

catboost_auc = roc_auc_score(
    y_valid,
    catboost_pred
)

print(f"CatBoost AUC: {catboost_auc:.6f}")


# ============================================================
# 11. Compare models
# ============================================================

results = pd.DataFrame(
    {
        "Model": [
            "Random Forest",
            "LightGBM",
            "CatBoost"
        ],
        "Validation AUC": [
            rf_auc,
            lgbm_auc,
            catboost_auc
        ]
    }
)

results = results.sort_values(
    by="Validation AUC",
    ascending=False
)

print("\n====================================")
print("Model comparison")
print("====================================")

print(results.to_string(index=False))


# ============================================================
# 12. Train LightGBM again using all available training data
# ============================================================

print("\n====================================")
print("Training final LightGBM...")
print("====================================")

lgbm_pipeline.fit(
    X,
    y
)


# ============================================================
# 13. Predict test set
# ============================================================

test_pred = lgbm_pipeline.predict_proba(
    X_test
)[:, 1]


# ============================================================
# 14. Create Kaggle submission
# ============================================================

submission = sample_submission.copy()

submission[TARGET] = test_pred

submission.to_csv(
    "submission_smoke_test.csv",
    index=False
)

print("\nSubmission created:")
print(submission.head())

print("\nSaved as:")
print("submission_smoke_test.csv")

ModuleNotFoundError: No module named 'lightgbm'